# Tugas Data Visualization – Analisis & Visualisasi Data Penjualan
**Nama:** Marcelino Jastin Ohoitimur  
**NIM:** 2418032  

Notebook ini mengikuti alur pada PPT: membaca data kotor, memeriksa masalah, membersihkan data, membuat KPI, menghasilkan 9 visualisasi, dan menampilkan insight utama.

**Cara pakai:** Jalankan cell dari atas ke bawah. Saat diminta, upload `dataset_penjualan_kotor.xlsx`. Jika ingin membandingkan hasilnya dengan data bersih, upload juga `dataset_penjualan_bersih.xlsx`.

## 1. Import library

In [ ]:
import os
import re
import difflib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.titleweight"] = "bold"
pd.options.display.float_format = "{:,.0f}".format

print("Library berhasil dimuat.")

## 2. Upload dan membaca dataset

In [ ]:
from google.colab import files

uploaded = files.upload()  # Pilih dataset_penjualan_kotor.xlsx dan, jika ada, dataset_penjualan_bersih.xlsx

FILE_KOTOR = "dataset_penjualan_kotor.xlsx"
FILE_BERSIH = "dataset_penjualan_bersih.xlsx"

if FILE_KOTOR not in uploaded and not os.path.exists(FILE_KOTOR):
    raise FileNotFoundError("Upload file dataset_penjualan_kotor.xlsx terlebih dahulu.")

df_kotor = pd.read_excel(FILE_KOTOR)
print(f"Ukuran data kotor: {df_kotor.shape[0]} baris x {df_kotor.shape[1]} kolom")
display(df_kotor.head(10))

print("=== Info tipe data ===")
df_kotor.info()
print("\n=== Jumlah nilai kosong per kolom ===")
print(df_kotor.isna().sum())
print("\n=== ID transaksi duplikat ===")
display(df_kotor[df_kotor["ID_Transaksi"].duplicated(keep=False)])

## 3. Melihat nilai kategori sebelum dibersihkan

In [ ]:
for col in ["Kategori", "Metode_Pembayaran", "Status"]:
    print(f"{col}: {sorted(df_kotor[col].astype(str).unique())}")

df = df_kotor.copy()

## 4. Data cleaning: ID transaksi dan tanggal

In [ ]:
# Membuat ID transaksi unik dan berurutan
# Catatan: cara ini mengganti semua ID mengikuti urutan baris.
df["ID_Transaksi"] = [f"TRX{i:03d}" for i in range(1, len(df) + 1)]

BULAN = {"januari":1, "februari":2, "maret":3, "april":4, "mei":5, "juni":6,
         "juli":7, "agustus":8, "september":9, "oktober":10, "november":11, "desember":12}

def parse_tanggal(x):
    if pd.isna(x):
        return pd.NaT
    if isinstance(x, (pd.Timestamp, np.datetime64)):
        return pd.to_datetime(x, errors="coerce")
    s = str(x).strip()
    try:
        m = re.fullmatch(r"(\d{1,2}) ([A-Za-z]+) (\d{4})", s)
        if m and m.group(2).lower() in BULAN:
            return pd.Timestamp(int(m.group(3)), BULAN[m.group(2).lower()], int(m.group(1)))
        m = re.fullmatch(r"(\d{4})[-/](\d{1,2})[-/](\d{1,2})", s)
        if m:
            return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))
        m = re.fullmatch(r"(\d{1,2})[-/](\d{1,2})[-/](\d{2,4})", s)
        if m:
            y = int(m.group(3))
            if y < 100:
                y += 2000
            return pd.Timestamp(y, int(m.group(2)), int(m.group(1)))
    except (ValueError, KeyError):
        pass
    return pd.NaT

df["Tanggal"] = df["Tanggal"].apply(parse_tanggal)
n_tgl_invalid = int(df["Tanggal"].isna().sum())

# Mengisi tanggal tidak valid berdasarkan tanggal terdekat.
# Jika ada tanggal sebelumnya dan sesudahnya, gunakan tanggal setelah tanggal sebelumnya.
for i in df.index[df["Tanggal"].isna()]:
    prev_dates = df.loc[:i-1, "Tanggal"].dropna() if i > df.index.min() else pd.Series(dtype="datetime64[ns]")
    next_dates = df.loc[i+1:, "Tanggal"].dropna() if i < df.index.max() else pd.Series(dtype="datetime64[ns]")
    if not prev_dates.empty:
        kandidat = prev_dates.iloc[-1] + pd.Timedelta(days=1)
    elif not next_dates.empty:
        kandidat = next_dates.iloc[0] - pd.Timedelta(days=1)
    else:
        kandidat = pd.Timestamp("2026-09-01") + pd.Timedelta(days=int(i))
    df.at[i, "Tanggal"] = kandidat

print(f"Tanggal tidak valid yang diperbaiki: {n_tgl_invalid}")

## 5. Data cleaning: nama produk, kategori, dan jumlah

In [ ]:
PRODUK_STANDAR = ["Laptop ASUS", "Mouse Logitech", "Keyboard Mechanical", "Monitor LG",
                  "Flashdisk 64GB", "Headset", "Kabel USB", "Webcam", "Printer Epson", "Tinta Printer"]
peta_lower = {p.lower(): p for p in PRODUK_STANDAR}

def rapikan_produk(x):
    s = " ".join(str(x).split()).lower()
    if s in peta_lower:
        return peta_lower[s]
    cocok = difflib.get_close_matches(s, list(peta_lower), n=1, cutoff=0.8)
    return peta_lower[cocok[0]] if cocok else str(x).strip().title()

df["Nama_Produk"] = df["Nama_Produk"].apply(rapikan_produk)
df["Kategori"] = df["Kategori"].astype(str).str.strip().str.capitalize()
peta_kategori = df.groupby("Nama_Produk")["Kategori"].agg(lambda s: s.mode().iat[0] if not s.mode().empty else "Lainnya")
df["Kategori"] = df["Nama_Produk"].map(peta_kategori).fillna(df["Kategori"])

KATA = {"satu":1, "dua":2, "tiga":3, "empat":4, "lima":5,
        "enam":6, "tujuh":7, "delapan":8, "sembilan":9, "sepuluh":10}
def ke_jumlah(x):
    if pd.isna(x):
        return 0
    s = str(x).strip().lower()
    if s in KATA:
        return KATA[s]
    angka = re.sub(r"[^0-9.\-]", "", s)
    return int(float(angka)) if angka else 0

df["Jumlah"] = df["Jumlah"].apply(ke_jumlah).abs()
display(df[["Nama_Produk", "Kategori", "Jumlah"]].head(10))

## 6. Data cleaning: harga, total penjualan, pelanggan, dan pembayaran

In [ ]:
def ke_angka(x):
    if pd.isna(x):
        return np.nan
    s = re.sub(r"[^\d\-]", "", str(x))  # menghapus Rp, titik pemisah ribuan, dan spasi
    return float(s) if s else np.nan

df["Harga_Satuan"] = df["Harga_Satuan"].apply(ke_angka)
harga_standar = df.groupby("Nama_Produk")["Harga_Satuan"].agg(lambda s: s.mode().iat[0] if not s.mode().empty else np.nan)
df["Harga_Satuan"] = df["Harga_Satuan"].fillna(df["Nama_Produk"].map(harga_standar))
# Jika masih ada harga kosong, isi dengan median harga yang tersedia.
df["Harga_Satuan"] = df["Harga_Satuan"].fillna(df["Harga_Satuan"].median()).fillna(0)

total_lama = df["Total_Penjualan"].apply(ke_angka)
df["Total_Penjualan"] = df["Jumlah"] * df["Harga_Satuan"]
n_total_diperbaiki = int((total_lama != df["Total_Penjualan"]).sum())
print(f"Total penjualan yang kosong/salah dan dikoreksi: {n_total_diperbaiki}")

def rapikan_pelanggan(x):
    s = " ".join(str(x).split()) if pd.notna(x) else ""
    return "Tidak Diketahui" if s.upper() in ("", "NAN", "N/A", "-") else s.title()

def rapikan_bayar(x):
    s = str(x).strip().lower()
    return {"tf":"Transfer", "transfer":"Transfer", "tunai":"Cash", "cash":"Cash", "qris":"QRIS"}.get(s, s.capitalize())

df["Nama_Pelanggan"] = df["Nama_Pelanggan"].apply(rapikan_pelanggan)
df["Metode_Pembayaran"] = df["Metode_Pembayaran"].apply(rapikan_bayar)
df["Status"] = df["Status"].astype(str).str.strip().str.capitalize()
df["Jumlah"] = df["Jumlah"].astype(int)
df["Harga_Satuan"] = df["Harga_Satuan"].round().astype(int)
df["Total_Penjualan"] = df["Total_Penjualan"].round().astype(int)

df_bersih = df.copy()
print("Data cleaning selesai.")
display(df_bersih.head(10))

## 7. Memeriksa hasil data bersih

In [ ]:
print("Ukuran data bersih :", df_bersih.shape)
print("Nilai kosong       :", int(df_bersih.isna().sum().sum()))
print("ID duplikat        :", int(df_bersih["ID_Transaksi"].duplicated().sum()))
print("Jumlah <= 0        :", int((df_bersih["Jumlah"] <= 0).sum()))
print("Total != Jml x Hrg :", int((df_bersih["Total_Penjualan"] != df_bersih["Jumlah"] * df_bersih["Harga_Satuan"]).sum()))
print("Rentang tanggal    :", df_bersih["Tanggal"].min().date(), "s/d", df_bersih["Tanggal"].max().date())
for col in ["Kategori", "Metode_Pembayaran", "Status"]:
    print(f"{col}: {sorted(df_bersih[col].astype(str).unique())}")
print("Produk:", sorted(df_bersih["Nama_Produk"].unique()))
display(df_bersih.dtypes)

df_bersih.to_excel("dataset_penjualan_bersih_hasil_colab.xlsx", index=False)
print("File hasil cleaning disimpan sebagai dataset_penjualan_bersih_hasil_colab.xlsx")

## 8. KPI / ringkasan penjualan

In [ ]:
total_omzet = df_bersih["Total_Penjualan"].sum()
jml_transaksi = len(df_bersih)
rata_transaksi = df_bersih["Total_Penjualan"].mean()
produk_terlaris = df_bersih.groupby("Nama_Produk")["Jumlah"].sum().idxmax()
produk_omzet_tertinggi = df_bersih.groupby("Nama_Produk")["Total_Penjualan"].sum().idxmax()

print(f"Total omzet                 : Rp {total_omzet:,.0f}".replace(",", "."))
print(f"Jumlah transaksi            : {jml_transaksi}")
print(f"Rata-rata nilai transaksi   : Rp {rata_transaksi:,.0f}".replace(",", "."))
print(f"Produk paling banyak terjual: {produk_terlaris}")
print(f"Produk omzet tertinggi      : {produk_omzet_tertinggi}")

omzet_produk = df_bersih.groupby("Nama_Produk")["Total_Penjualan"].sum().sort_values(ascending=True)
unit_produk = df_bersih.groupby("Nama_Produk")["Jumlah"].sum().sort_values(ascending=False)
omzet_kat = df_bersih.groupby("Kategori")["Total_Penjualan"].sum()
bayar = df_bersih["Metode_Pembayaran"].value_counts()

## 9. Grafik 1 dan 2 – Omzet dan unit terjual per produk

In [ ]:
fig, ax = plt.subplots()
bars = ax.barh(omzet_produk.index, omzet_produk.values / 1e6, color=sns.color_palette("viridis", len(omzet_produk)))
ax.set_title("Total Omzet per Produk")
ax.set_xlabel("Omzet (juta Rupiah)")
ax.set_ylabel("")
for b in bars:
    ax.text(b.get_width() + 0.3, b.get_y() + b.get_height()/2, f"{b.get_width():,.1f} jt", va="center", fontsize=9)
ax.set_xlim(0, max(omzet_produk.max()/1e6 * 1.15, 1))
plt.tight_layout(); plt.show()

fig, ax = plt.subplots()
sns.barplot(x=unit_produk.index, y=unit_produk.values, hue=unit_produk.index, legend=False, palette="crest", ax=ax)
ax.set_title("Jumlah Unit Terjual per Produk")
ax.set_xlabel(""); ax.set_ylabel("Unit")
plt.xticks(rotation=35, ha="right")
for i, v in enumerate(unit_produk.values):
    ax.text(i, v + 0.5, str(v), ha="center", fontsize=9)
plt.tight_layout(); plt.show()

## 10. Grafik 3 dan 4 – Proporsi kategori dan tren mingguan

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(omzet_kat, labels=omzet_kat.index, autopct="%1.1f%%", startangle=90,
       colors=sns.color_palette("Set2"), wedgeprops=dict(width=0.45, edgecolor="white"))
ax.set_title("Proporsi Omzet per Kategori")
plt.show()

tren = df_bersih.set_index("Tanggal")["Total_Penjualan"].resample("W").sum() / 1e6
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(tren.index, tren.values, marker="o", linewidth=2, color="#2a9d8f")
ax.fill_between(tren.index, tren.values, alpha=0.15, color="#2a9d8f")
ax.set_title("Tren Omzet Mingguan"); ax.set_xlabel("Minggu"); ax.set_ylabel("Omzet (juta Rupiah)")
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

## 11. Grafik 5 – Jumlah transaksi dan omzet per metode pembayaran

In [ ]:
omzet_bayar = df_bersih.groupby("Metode_Pembayaran")["Total_Penjualan"].sum() / 1e6
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(x=bayar.index, y=bayar.values, hue=bayar.index, legend=False, palette="Set2", ax=axes[0])
axes[0].set_title("Jumlah Transaksi per Metode Pembayaran")
axes[0].set_xlabel(""); axes[0].set_ylabel("Jumlah transaksi")
for i, v in enumerate(bayar.values):
    axes[0].text(i, v + 0.3, str(v), ha="center")
sns.barplot(x=omzet_bayar.index, y=omzet_bayar.values, hue=omzet_bayar.index, legend=False, palette="Set2", ax=axes[1])
axes[1].set_title("Omzet per Metode Pembayaran")
axes[1].set_xlabel(""); axes[1].set_ylabel("Omzet (juta Rupiah)")
plt.tight_layout(); plt.show()

## 12. Grafik 6 dan 7 – Top 10 pelanggan dan heatmap

In [ ]:
top_pel = (df_bersih[df_bersih["Nama_Pelanggan"] != "Tidak Diketahui"]
           .groupby("Nama_Pelanggan")["Total_Penjualan"].sum()
           .sort_values(ascending=False).head(10)[::-1])
fig, ax = plt.subplots()
ax.barh(top_pel.index, top_pel.values / 1e6, color=sns.color_palette("rocket_r", len(top_pel)))
ax.set_title("Top 10 Pelanggan (Total Belanja)"); ax.set_xlabel("Total belanja (juta Rupiah)")
plt.tight_layout(); plt.show()

pivot = df_bersih.pivot_table(index="Nama_Produk", columns="Metode_Pembayaran", values="Total_Penjualan", aggfunc="sum", fill_value=0) / 1e6
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlGnBu", linewidths=.5,
            cbar_kws={"label":"Omzet (juta Rp)"}, ax=ax)
ax.set_title("Omzet (juta Rp): Produk × Metode Pembayaran")
ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout(); plt.show()

## 13. Grafik 8 dan 9 – Sebaran transaksi dan omzet bulanan

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df_bersih, x="Kategori", y="Total_Penjualan", hue="Kategori", legend=False, palette="Set2", ax=ax)
ax.set_title("Sebaran Nilai Transaksi per Kategori")
ax.set_xlabel(""); ax.set_ylabel("Total penjualan (Rp)")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x/1e6:.0f} jt"))
plt.tight_layout(); plt.show()

bulanan = (df_bersih.assign(Bulan=df_bersih["Tanggal"].dt.to_period("M").astype(str))
           .pivot_table(index="Bulan", columns="Kategori", values="Total_Penjualan", aggfunc="sum", fill_value=0) / 1e6)
ax = bulanan.plot(kind="bar", stacked=True, figsize=(9, 5), color=sns.color_palette("Set2"))
ax.set_title("Omzet Bulanan per Kategori"); ax.set_xlabel("Bulan"); ax.set_ylabel("Omzet (juta Rupiah)")
plt.xticks(rotation=0); plt.legend(title="Kategori"); plt.tight_layout(); plt.show()

## 14. Insight utama dan (opsional) perbandingan file data bersih

In [ ]:
kat_top = omzet_kat.idxmax()
bayar_top = bayar.idxmax()
bln_top = bulanan.sum(axis=1).idxmax()
print("INSIGHT UTAMA")
print("-" * 60)
print(f"1. Total omzet {jml_transaksi} transaksi mencapai Rp {total_omzet:,.0f}.".replace(",", "."))
print(f"2. Omzet terbesar berasal dari produk '{produk_omzet_tertinggi}', sedangkan unit terbanyak terjual adalah '{produk_terlaris}'.")
print(f"3. Kategori '{kat_top}' menyumbang {omzet_kat.max()/omzet_kat.sum():.1%} dari total omzet.")
print(f"4. Metode pembayaran paling sering dipakai: {bayar_top} ({bayar.max()} transaksi).")
print(f"5. Bulan dengan omzet tertinggi: {bln_top}.")

# Jika dataset bersih juga di-upload, bandingkan jumlah baris dan total omzetnya.
if os.path.exists(FILE_BERSIH):
    df_referensi = pd.read_excel(FILE_BERSIH)
    print("\nPERBANDINGAN DENGAN FILE DATA BERSIH YANG DIUPLOAD")
    print("Baris hasil cleaning :", len(df_bersih))
    print("Baris data referensi :", len(df_referensi))
    print("Omzet hasil cleaning : Rp", f"{df_bersih['Total_Penjualan'].sum():,.0f}".replace(",", "."))
    if "Total_Penjualan" in df_referensi.columns:
        print("Omzet data referensi : Rp", f"{pd.to_numeric(df_referensi['Total_Penjualan'], errors='coerce').sum():,.0f}".replace(",", "."))

print("\nUntuk mengunduh file hasil cleaning, jalankan cell berikutnya jika diperlukan.")

## 15. Download hasil data bersih (opsional)

In [ ]:
from google.colab import files
files.download("dataset_penjualan_bersih_hasil_colab.xlsx")